In [1]:
import time
from stable_baselines3 import PPO

from spotmicro.env.spotmicro_env import SpotmicroEnv
from spotmicro.physics.factory import create_backend
from spotmicro.devices.fixed_controller import FixedController
from spotmicro.tools.config import Config
from reward_function import reward_function, RewardState

pybullet build time: Apr  4 2025 18:56:19


In [ ]:
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.logger import configure

# ========= CONFIG ==========
TOTAL_STEPS = 2_000_000
run = "standPB"
log_dir = f"./logs/{run}"

def clipped_linear_schedule(initial_value, min_value=1e-5):
    def schedule(progress_remaining):
        return max(progress_remaining * initial_value, min_value)
    return schedule

checkpoint_callback = CheckpointCallback(
    save_freq=TOTAL_STEPS // 5,
    save_path=f"{run}_checkpoints",
    name_prefix=f"ppo_{run}"
)

# ========= ENV ==========
cfg = Config()
dev = FixedController("still") #not a configurable class
backend = create_backend("pybullet", use_gui=False)
env = SpotmicroEnv(
    backend,
    dev,
    cfg,
    reward_function,
    RewardState(),
    use_gui=False
)
check_env(env, warn=True)


# ========= MODEL ==========
model = PPO(
    "MlpPolicy", 
    env,
    verbose=1,   # no default printouts
    learning_rate=clipped_linear_schedule(3e-4),
    ent_coef=0.001,
    clip_range=0.1,
    tensorboard_log=log_dir,
    device = 'cpu'
)

# Custom logger: ONLY csv + tensorboard (no stdout table)
new_logger = configure(log_dir, ["csv", "tensorboard"])
model.set_logger(new_logger)

# ========= TRAIN ==========
model.learn(
    total_timesteps=TOTAL_STEPS,
    reset_num_timesteps=False,
    callback=checkpoint_callback
)
model.save(f"ppo_{run}")
env.close()

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [ ]:
policy = "standPB"

cfg = Config()
dev = FixedController("still")
backend = create_backend("mujoco", use_gui=True)
env = SpotmicroEnv(
    backend,
    dev,
    cfg,
    reward_function,
    RewardState(),
    use_gui=True
)
obs, _ = env.reset()

# === Load model ===
model = PPO.load(f"ppo_{policy}", device = 'cpu')
#model = PPO.load(f"{policy}_checkpoints/ppo_{policy}_3000000_steps")
base_steps = env.num_steps

for joint in env.agent.motor_joints:
    print(f"{joint.name}: {joint.limits}")

t0 = time.time()
for _ in range(3001):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        print("Terminated")
        env.plot_reward_components()  # plot per episode
        obs, _ = env.reset()
        print(f"Num steps: {env.num_steps - base_steps}")
        break
    
t1 = time.time()
print(f"Elapsed real time: {t1-t0}")

env.close()

front_left_shoulder: (-0.548, 0.548)
front_left_leg: (-2.666, 1.548)
front_left_foot: (-0.1, 2.59)
front_right_shoulder: (-0.548, 0.548)
front_right_leg: (-2.666, 1.548)
front_right_foot: (-0.1, 2.59)
rear_left_shoulder: (-0.548, 0.548)
rear_left_leg: (-2.666, 1.548)
rear_left_foot: (-0.1, 2.59)
rear_right_shoulder: (-0.548, 0.548)
rear_right_leg: (-2.666, 1.548)
rear_right_foot: (-0.1, 2.59)
